# Vision Transformers (ViT)

_Modern Deep Learning & AI_

**Chop an image into patches, treat each patch as a word, and let attention mix them.**

Transformers conquered language by treating a sentence as a sequence of word tokens. A Vision Transformer does the same to images.

     It slices the image into a grid of small square patches. Each patch becomes one token, like a word.

---

This notebook builds a Vision Transformer from scratch, one piece at a time. Run each cell, read the note above it, and watch how patches, the CLS token, and attention fit together. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Define the Vision Transformer

A Transformer expects a sequence of tokens, so the model chops the image into a grid of patches and embeds each into a vector of length `dim`. A single strided convolution does this in one shot — a `patch`-sized kernel with `stride=patch` lands on each patch exactly once, which is the same as flattening every patch and passing it through one linear layer. We add a learned `[CLS]` token to summarize the whole image and a positional embedding so the model knows where each patch sat; the forward pass then runs the tokens through `depth` self-attention layers and classifies from the CLS token, since it attended to every patch.

In [ ]:
import torch
import torch.nn as nn

class ViT(nn.Module):
    def __init__(self, img=32, patch=8, dim=64, depth=2, heads=4, n_cls=10):
        super().__init__()
        n_patches = (img // patch) ** 2                  # N = (H/P)*(W/P)
        # one strided conv = flatten-each-patch then linear-embed
        self.embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))  # [CLS] token
        self.pos = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        enc = nn.TransformerEncoderLayer(dim, heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=depth)
        self.head = nn.Linear(dim, n_cls)

    def forward(self, x):                                 # x: (B, 3, H, W)
        t = self.embed(x).flatten(2).transpose(1, 2)     # (B, N, dim) tokens
        cls = self.cls.expand(x.size(0), -1, -1)
        t = torch.cat([cls, t], dim=1) + self.pos        # prepend CLS, add pos
        t = self.encoder(t)                              # attention over patches
        return self.head(t[:, 0])                        # classify from CLS

### Step 2 — Run a batch through the model

With the model defined, we push two synthetic RGB images through it to check the whole pipeline. The output is one score per class for each image, so we expect a `(batch, n_cls)` tensor. Printing its shape confirms the patch-to-token-to-CLS wiring lines up.

In [ ]:
model = ViT()
x = torch.randn(2, 3, 32, 32)        # 2 synthetic RGB images
logits = model(x)                    # (2, 10) class scores
print(logits.shape)

## Visualize the data & results

_On a real handwritten 3 from load_digits(), which image patches does the CLS token attend to most?_

### Step 1 — Slice a real digit into 3x3 patches

To see attention concretely we take one real handwritten `3` and tile it the same way the ViT does. The 8x8 image is padded to 9x9 so it divides evenly into a 3x3 grid of 3x3-pixel patches. Flattening each patch gives nine little vectors — the per-patch tokens we will attend over.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.data[digits.target == 3][0].reshape(8, 8)   # a real handwritten 3
imgp = np.pad(img, ((0, 1), (0, 1)), mode='edge')        # 9x9 so it tiles 3x3

# 9 patches of 3x3 pixels each.
patches = np.array([imgp[r*3:r*3+3, c*3:c*3+3].flatten()
                    for r in range(3) for c in range(3)], dtype=float)

### Step 2 — Compute CLS attention with scaled dot-product softmax

Attention scores each patch by how similar it is to a query vector. We normalize every patch token to unit length, build the CLS query as the (normalized) average patch, and take scaled dot products between the query and each patch. A softmax turns those scores into nine weights that sum to 1 — the share of attention the CLS token pays to each patch.

In [ ]:
pn = patches / (np.linalg.norm(patches, axis=1, keepdims=True) + 1e-9)
q = pn.mean(0)                                           # CLS query (mean patch)
q = q / (np.linalg.norm(q) + 1e-9)

scores = pn @ q * 4.0                                    # scaled dot-product
w = np.exp(scores - scores.max())                        # softmax numerator
w = w / w.sum()                                          # normalize -> sums to 1
grid = w.reshape(3, 3)

### Step 3 — Plot the attention map

Left: the 3x3 attention grid laid over the digit's patch layout, with each cell's weight printed on it — brighter means the CLS token looked there more. Right: the same nine weights as a bar chart, with the single most-attended patch highlighted. The strokes of the `3` draw the most attention, which is exactly the signal a classifier would want.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

# Left — attention heat-map over the 3x3 patch grid.
im = ax[0].imshow(grid, cmap='magma')
ax[0].set_title('CLS attention over 3x3 patches of a real 3')
ax[0].set_xticks(range(3))
ax[0].set_yticks(range(3))
for i in range(3):
    for j in range(3):
        ax[0].text(j, i, format(grid[i, j], '.3f'), ha='center', va='center', color='w')
fig.colorbar(im, ax=ax[0])

# Right — attention weight per patch token, top patch highlighted.
top_patch = int(np.argmax(w))
cols = ['#ffb454' if i == top_patch else '#4ea1ff' for i in range(9)]
ax[1].bar([f'p{i}' for i in range(9)], w, color=cols)
ax[1].set_title('Attention weight per patch token')
ax[1].set_ylabel('weight')

plt.tight_layout()
plt.show()